In [1]:
import os
import torch
import rlcard
from rlcard.agents.dmc_agent import DMCTrainer
from rlcard.envs.registration import register
from rlcard.agents.dmc_agent.model import DMCModel
from rlcard.agents import RandomAgent
from rlcard.utils import tournament
import math

In [2]:
register(env_id='tien-len', entry_point='envs.tienlen_env:TienLenEnv')
env = rlcard.make('tien-len')


In [ ]:
# Agents
class TrainedAgent:
    def __init__(self, model):
        self.model = model
    def step(self, state):
        # Option 2: Score all legal action features and pick the max
        obs = torch.FloatTensor(state['obs']).unsqueeze(0)
        legal_actions = state['legal_actions']
        best_action, max_q = None, -float('inf')

        for action, feature in legal_actions.items():
            feat = torch.FloatTensor(feature).unsqueeze(0)
            with torch.no_grad():
                q_value = self.model(obs, feat)
            if q_value > max_q:
                max_q = q_value
                best_action = action
        return best_action

    def eval_step(self, state):
        return self.step(state), {}

class MoveSequence(list):
    def __init__(self, score, sequence_or_node):
        if isinstance(sequence_or_node, list):
            super().__init__(sequence_or_node)
        else:
            super().__init__([sequence_or_node])
        self.score = score

class HeuristicSearchAgent:
    def __init__(self, judger, pruning_threshold=0.2):
        self.judger = judger
        self.PRUNING_THRESHOLD = pruning_threshold

    def evaluate_move(self, hand, move_played, state, moves):
        score = 10 * math.log(1 + len(moves))
        if not hand: score += 1000
        score += math.pow(1.5, (13 - len(hand)))

        # Mobility and positional strength
        for r, s in hand:
            score += ((r * 4 + s) / len(hand)) * 5
        if moves:
            score += sum(len(m) for m in moves) / len(moves)

        move_type, _ = self.judger.get_type(move_played)
        if move_played:
            # Check mobility after this move
            future_moves = self.judger._get_all_types(hand)
            for m in future_moves:
                score += math.log(len(m)) if move_type == "NONE" else len(m)

        if move_type == "HANG" and state in ["PIG", "SAME"]:
            score += 250
        return score

    def minimax(self, hand, move, state, depth, acc_score, best_score):
        hand_copy = [c for c in hand if c not in move]
        new_state, _ = self.judger.get_type(move)
        next_moves = self.judger._get_all_types(hand_copy)

        curr_score = self.evaluate_move(hand_copy, move, state, next_moves)

        # Pruning and Base Case
        if depth == 0 or not next_moves or curr_score < best_score * self.PRUNING_THRESHOLD:
            final_score = acc_score + (curr_score / (0.9**depth)) * (depth + 1)
            return MoveSequence(final_score, move)

        max_val = -float('inf')
        best_seq = []
        for nxt in next_moves:
            eval_seq = self.minimax(hand_copy, nxt, new_state, depth-1,
                                    acc_score + curr_score/(0.9**depth), max_val)
            if eval_seq.score > max_val:
                max_val = eval_seq.score
                best_seq = list(eval_seq)

        best_seq.insert(0, move)
        return MoveSequence(max_val, best_seq)

    def step(self, state_obs):
        hand = state_obs['raw_obs']['hand']
        stack_state = state_obs['raw_obs']['state']
        legal_moves = [m for m in state_obs['raw_legal_actions'] if m]

        if not legal_moves: return ()

        best_move, best_val = None, None
        for m in legal_moves:
            res = self.minimax(hand, m, stack_state, 2, 0, -float('inf'))
            res.score /= 3.0 # depth + 1
            if best_val is None or res.score > best_val.score:
                best_val, best_move = res, m

        # Pass if "doing nothing" is better than the best move found
        baseline = self.evaluate_move(hand, (), "NONE", self.judger._get_all_types(hand))
        return best_move if (best_val and best_val.score > baseline) else ()

    def eval_step(self, state):
        return self.step(state), {}

class NaiveTienLenAgent:
    """
    Naive implementation: Always plays most cards possible, smallest first.
    If an opponent is about to win, it switches to playing high cards.
    """
    def __init__(self):
        pass

    def step(self, state):
        legal_actions = state['raw_legal_actions'] # List of card tuples
        if not legal_actions or legal_actions == [()]:
            return ()

        # Remove 'Pass' from consideration for sorting logic
        playable = [a for a in legal_actions if a != ()]
        if not playable: return ()

        # Java logic: Sort by size (desc), then rank (asc), then suit (desc)
        # Note: action is a tuple of (rank, suit)
        playable.sort(key=lambda move: (
            -len(move),               # sizeCompare (descending)
            move[0][0],               # rankCompare (ascending)
            -move[-1][1]              # suitCompare (descending)
        ))

        # Check if someone is about to win (Java: someoneAboutToWin)
        # state['others_hand_count'] from our Game.get_state
        someone_about_to_win = any(count == 1 for count in state['raw_obs']['others_hand_count'])

        # Java logic: If same state and someone winning, play the highest legal same-rank combo
        if state['raw_obs']['state'] == "SAME" and someone_about_to_win:
            return playable[-1] # Smallest index in Java sorted list was highest rank?
                               # In Python, [-1] is the highest rank/suit.

        return playable[0]

    def eval_step(self, state):
        return self.step(state), {}


In [ ]:
def train():
    env = rlcard.make('tien-len')
    trainer = DMCTrainer(
        env,
        cuda='0',
        xpid='rl-tien_len',
        savedir='models',
        num_actors=5,
    )
    trainer.start()

def load_model():
    env = rlcard.make('tien-len-v1')

    # 2. Re-create the model structure and load weights
    # Ensure state_shape and action_shape match your Env
    model = DMCModel(env.state_shape[0], env.action_shape[0])
    checkpoint = torch.load('experiments/tien_len_v1/model.tar', map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # Set to evaluation mode

    return TrainedAgent(model)


def evaluate(trained_agent):
    env.set_agents([trained_agent, NaiveTienLenAgent(), HeuristicSearchAgent, RandomAgent(num_actions=env.num_actions)])
    results = tournament(env, 1000)
    print(f"Tournament Results (Avg Payoffs): {results}")


def play_test_game(trained_agent):
    env = rlcard.make('tien-len-v1')
    env.set_agents([trained_agent, NaiveTienLenAgent(), HeuristicSearchAgent, RandomAgent(num_actions=env.num_actions)])

    state, player_id = env.reset()
    print("--- Game Start ---")

    while not env.is_over():
        # Get action from the current player's agent
        current_agent = env.agents[player_id]
        action, _ = current_agent.eval_step(state)

        # Display the play
        move_desc = action if action else "PASS"
        print(f"Player {player_id} plays: {move_desc}")

        # Advance environment
        state, player_id = env.step(action)

    print("--- Game Over ---")
    print(f"Final Payoffs: {env.get_payoffs()}")


In [ ]:
if not os.path.exists('models'): os.makedirs('models')
train()

In [ ]:
evaluate()

In [ ]:
play_test_game()